# nb05 — Embedding Model Evaluation

**Goal.** Pick a production retriever for the Kalisio RAG pipeline. The corpus is
English-only, but most queries arrive in French — cross-lingual retrieval has to be
measured empirically.

**What this notebook does.** Score three dense models against a hand-built gold set,
compare them to BM25, fuse the best dense run with BM25 (RRF, K=60), then rerank the
top-20 with a cross-encoder. Layers and metrics:

| Layer       | Query style                  | What it tests                 |
|-------------|------------------------------|-------------------------------|
| `A_symbol`  | API / component name         | exact identifier match        |
| `B_docs`    | Natural-language question    | semantic understanding        |
| `C_code`    | Concept → file               | cross-modal (NL → code)       |
| `negative`  | Out-of-scope question        | rejection                     |

Gold lives in `outputs/nb05_gold.json` (authored under
`experiments/nb05_embedding_eval/gold_draft.json`).

**Lineup.** Earlier runs swept seven embedding recipes; the four that did not earn
their cost are dropped here:
- `e5-large` / `e5-large-instruct` — 512-token cap forces truncation, no win on FR.
- `nomic-v1.5` — English-only, collapses on FR `B_docs` / `C_code`.
- `jina-code` — incompatible with the current `transformers` pin.

What remains:
- `qwen3-0.6b` — multilingual + code-aware (top of the prior leaderboard).
- `arctic-l-v2` — strong multilingual baseline.
- `bge-m3` — multilingual reference, long context.

In [1]:
import os, sys, gc, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "outputs" / "nb05_gold.json").exists():
            return candidate
        knowledge = candidate / "knowledge"
        if (knowledge / "pyproject.toml").exists() and (knowledge / "outputs" / "nb05_gold.json").exists():
            return knowledge
    raise FileNotFoundError("Could not find knowledge project root with outputs/nb05_gold.json")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "experiments" / "nb05_embedding_eval"))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[setup] device={DEVICE}", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
GOLD_PATH = OUTPUTS / "nb05_gold.json"

[setup] device=cuda NVIDIA GeForce RTX 3060 Ti


## 1. Load corpus and gold

In [2]:
from corpus_filter import scan_corpus
from corpus_filter.models import FilterConfig
from corpus_filter.profiles import build_js_vue_rag_profile
from chunking import chunk_files
from nb05_helpers import load_gold_validated, gold_summary

_base = build_js_vue_rag_profile()
cfg = FilterConfig(
    excluded_dirs=_base.excluded_dirs - {"docs", "tools"},
    excluded_extensions=_base.excluded_extensions,
    excluded_filenames=_base.excluded_filenames,
    excluded_patterns=_base.excluded_patterns,
    max_file_size=_base.max_file_size,
    max_line_length=_base.max_line_length,
    included_extensions={".md", ".js", ".mjs", ".vue", ".json"},
)
scan = scan_corpus(config=cfg)
chunks = chunk_files(scan.included)
chunk_texts   = [c["text"] for c in chunks]
chunk_sources = [c["metadata"]["source"] for c in chunks]
print(f"[corpus] files={len(scan.included)}  chunks={len(chunks)}")

queries = load_gold_validated(GOLD_PATH, chunk_sources)
query_en = [q.en for q in queries]
query_fr = [q.fr for q in queries]
print(f"[gold]   {gold_summary(queries)}")

[corpus] files=1251  chunks=9893
[gold]   {'A_symbol': 60, 'B_docs': 80, 'C_code': 45, 'negative': 15, 'total': 200}


In [3]:
print("Sample query per layer:\n")
seen = set()
for q in queries:
    if q.layer in seen:
        continue
    seen.add(q.layer)
    print(f"[{q.layer}] {q.id}")
    print(f"  EN: {q.en}")
    print(f"  FR: {q.fr}")
    print(f"  gold: {list(q.gold_sources)}\n")

Sample query per layer:

[A_symbol] A-001
  EN: addLayer function
  FR: fonction addLayer
  gold: ['kdk/docs/api/map/map-mixins.md', 'kdk/docs/api/map/globe-mixins.md']

[B_docs] B-001
  EN: How do I add a new layer to a map?
  FR: Comment ajouter une nouvelle couche à la carte ?
  gold: ['kdk/docs/api/map/map-mixins.md', 'kdk/docs/api/map/globe-mixins.md']

[C_code] C-001
  EN: Where is the addLayer logic implemented for the 2D map?
  FR: Où est implémentée la logique addLayer pour la carte 2D ?
  gold: ['kdk/core/client/mixins/mixin.service.js', 'kdk/map/client/mixins/map/mixin.base-map.js']

[negative] N-001
  EN: How do I integrate TensorFlow.js for ML predictions?
  FR: Comment intégrer TensorFlow.js pour des prédictions ML ?
  gold: []



## 2. Candidate models

Three dense recipes survive the cull. All three accept 8 K tokens, so no
truncation audit is needed — every chunk fits. Recipes also carry their
model-specific query/passage prefixes (E5-style `query:` / `passage:`,
Qwen3 instruct prompt, etc.).

In [4]:
from nb05_helpers import RECIPES

# Curated lineup — see notebook intro for why the others are dropped.
SELECTED_MODELS = ["qwen3-0.6b", "arctic-l-v2", "bge-m3"]
ACTIVE_RECIPES = {k: RECIPES[k] for k in SELECTED_MODELS}

display(pd.DataFrame([
    {
        "key": k,
        "model_id": r.model_id,
        "query_prefix": repr(r.query_prefix),
        "passage_prefix": repr(r.passage_prefix),
        "max_tokens": r.max_tokens,
        "family": r.family,
    } for k, r in ACTIVE_RECIPES.items()
]).set_index("key"))

,model_id,query_prefix,passage_prefix,max_tokens,family
key,,,,,
qwen3-0.6b,Qwen/Qwen3-Embedding-0.6B,"'Instruct: Given a developer question in French or English, retrieve the relevant Kalisio docume...",'',8192,multilingual-code-dense
arctic-l-v2,Snowflake/snowflake-arctic-embed-l-v2.0,'query: ','',8192,multilingual-dense
bge-m3,BAAI/bge-m3,'','',8192,multilingual-dense


## 3. Dense bake-off

Encode the corpus once per model, then score the EN and FR query sets
against gold. Output is one row per query × language × model — we collapse
it later. `hit@k` defaults to k=5; the helpers also record hit@1 / hit@10.

In [5]:
from nb05_helpers import (
    encode_corpus, encode_queries, load_recipe_model,
    dense_rank, evaluate_ranks,
)

dense_ranks: dict[str, tuple[np.ndarray, np.ndarray]] = {}
dense_dfs: list[pd.DataFrame] = []
available_models: list[str] = []
skipped_models: list[dict[str, str]] = []

for key, recipe in ACTIVE_RECIPES.items():
    print(f"[dense] {key} -> loading")
    model = None
    try:
        model = load_recipe_model(recipe)
        print(f"[dense] {key} -> encoding {len(chunks)} chunks")
        cv = encode_corpus(model, recipe, chunk_texts)
        en_v = encode_queries(model, recipe, query_en)
        fr_v = encode_queries(model, recipe, query_fr)

        en_r = dense_rank(en_v, cv)
        fr_r = dense_rank(fr_v, cv)
        dense_ranks[key] = (en_r, fr_r)
        available_models.append(key)

        dense_dfs.append(evaluate_ranks(en_r, queries, chunk_sources, language="en", approach=key))
        dense_dfs.append(evaluate_ranks(fr_r, queries, chunk_sources, language="fr", approach=key))
    except Exception as exc:
        skipped_models.append({"model": key, "error": type(exc).__name__, "message": str(exc)})
        print(f"[dense] SKIP {key}: {type(exc).__name__}: {exc}")
    finally:
        for var_name in ("model", "cv", "en_v", "fr_v"):
            if var_name in locals():
                del locals()[var_name]
        gc.collect()
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

if not dense_dfs:
    raise RuntimeError("No dense embedding model completed successfully.")

df_dense = pd.concat(dense_dfs, ignore_index=True)
print(f"\n[dense] available={available_models}")
if skipped_models:
    display(pd.DataFrame(skipped_models))
df_dense.head()

[dense] qwen3-0.6b -> loading


[dense] qwen3-0.6b -> encoding 9893 chunks
[dense] arctic-l-v2 -> loading
[dense] arctic-l-v2 -> encoding 9893 chunks
[dense] bge-m3 -> loading
[dense] bge-m3 -> encoding 9893 chunks

[dense] available=['qwen3-0.6b', 'arctic-l-v2', 'bge-m3']


,approach,language,layer,query_id,is_negative,hit@k,recall@k,mrr,hit@1,recall@1,hit@5,recall@5,hit@10,recall@10
0,qwen3-0.6b,en,A_symbol,A-001,False,0,0.0,0.029412,0,0.0,0,0.0,0,0.0
1,qwen3-0.6b,en,A_symbol,A-002,False,0,0.0,0.027027,0,0.0,0,0.0,0,0.0
2,qwen3-0.6b,en,A_symbol,A-003,False,1,1.0,0.500000,0,0.0,1,1.0,1,1.0
3,qwen3-0.6b,en,A_symbol,A-004,False,0,0.0,0.024390,0,0.0,0,0.0,0,0.0
4,qwen3-0.6b,en,A_symbol,A-005,False,0,0.0,0.023810,0,0.0,0,0.0,0,0.0


## 4. BM25 baseline

Lexical floor. Strong on `A_symbol` (identifiers are literal tokens),
weak everywhere else — especially on FR, where the query language
barely overlaps the EN corpus.

In [6]:
from nb05_helpers import bm25_rank

bm25_en_ranks = bm25_rank(chunk_texts, query_en)
bm25_fr_ranks = bm25_rank(chunk_texts, query_fr)
df_bm25 = pd.concat([
    evaluate_ranks(bm25_en_ranks, queries, chunk_sources, language="en", approach="bm25"),
    evaluate_ranks(bm25_fr_ranks, queries, chunk_sources, language="fr", approach="bm25"),
], ignore_index=True)
df_bm25.groupby(["layer", "language"])["hit@k"].mean().unstack("language").round(3)

language,en,fr
layer,,
A_symbol,0.850,0.833
B_docs,0.400,0.100
C_code,0.311,0.089
negative,0.333,0.000


## 5. Hybrid (Dense + BM25 via RRF, K=60)

Reciprocal-rank fusion of the leading dense run with BM25. Earlier sweeps
tried K ∈ {30, 60, 90} — K=60 won every time, so we lock it in here. The
leader is selected automatically from §3 (currently `qwen3-0.6b`).

In [7]:
from nb05_helpers import rrf_fuse

K_RRF = 60
df_hybrid = pd.DataFrame()
leader_key = None
if available_models:
    leader_key = df_dense.groupby("approach")["hit@k"].mean().idxmax()
    print(f"[hybrid] leader = {leader_key}, K = {K_RRF}")
    en_leader, fr_leader = dense_ranks[leader_key]
    en_fused = rrf_fuse(en_leader, bm25_en_ranks, k_rrf=K_RRF)
    fr_fused = rrf_fuse(fr_leader, bm25_fr_ranks, k_rrf=K_RRF)
    approach = f"hybrid_k{K_RRF}"
    df_hybrid = pd.concat([
        evaluate_ranks(en_fused, queries, chunk_sources, language="en", approach=approach),
        evaluate_ranks(fr_fused, queries, chunk_sources, language="fr", approach=approach),
    ], ignore_index=True)
else:
    print("[hybrid] no dense models available, skipping")

df_hybrid.groupby(["approach", "layer"])["hit@k"].mean().unstack("layer").round(3) if not df_hybrid.empty else df_hybrid

[hybrid] leader = qwen3-0.6b, K = 60


layer,A_symbol,B_docs,C_code,negative
approach,,,,
hybrid_k60,0.875,0.575,0.589,0.033


## 6. Cross-encoder reranker (Layer B + C only)

Rerank the top-20 of `hybrid_k60` with a cross-encoder. We only apply it
to natural-language layers (`B_docs`, `C_code`) — symbol queries already
saturate, and reranking negatives doesn't help rejection. A wider top-40
pool was tried in earlier runs and didn't move the leaderboard, so the
single top-20 setting is kept.

In [8]:
import importlib
import nb05_helpers
importlib.reload(nb05_helpers)
from nb05_helpers import load_reranker, rerank_topk

RERANK_TOPK = 20
RERANK_BATCH_SIZE = 4
df_rerank = pd.DataFrame()

if df_hybrid.empty:
    print("[reranker] skipped (no hybrid baseline)")
else:
    try:
        reranker = load_reranker()
    except Exception as exc:
        print(f"[reranker] unavailable: {type(exc).__name__}: {exc}")
        reranker = None

    if reranker is not None:
        print(f"[reranker] reranking top-{RERANK_TOPK} on top of hybrid_k{K_RRF}")
        en_leader, fr_leader = dense_ranks[leader_key]
        en_fused = rrf_fuse(en_leader, bm25_en_ranks, k_rrf=K_RRF)
        fr_fused = rrf_fuse(fr_leader, bm25_fr_ranks, k_rrf=K_RRF)

        rerank_eligible = [i for i, q in enumerate(queries) if q.layer in {"B_docs", "C_code"}]

        def rerank_one(fused: np.ndarray, q_text: str, qi: int, k_cand: int) -> np.ndarray:
            cand = fused[qi, :k_cand].tolist()
            order = rerank_topk(reranker, q_text, [chunks[c]["text"] for c in cand], batch_size=RERANK_BATCH_SIZE)
            reranked = [cand[o] for o in order]
            return np.array(reranked + fused[qi, k_cand:].tolist())

        en_reranked = en_fused.copy()
        fr_reranked = fr_fused.copy()
        for qi in rerank_eligible:
            en_reranked[qi] = rerank_one(en_fused, queries[qi].en, qi, RERANK_TOPK)
            fr_reranked[qi] = rerank_one(fr_fused, queries[qi].fr, qi, RERANK_TOPK)

        approach = f"hybrid_k{K_RRF}+rerank"
        df_rerank = pd.concat([
            evaluate_ranks(en_reranked, queries, chunk_sources, language="en", approach=approach),
            evaluate_ranks(fr_reranked, queries, chunk_sources, language="fr", approach=approach),
        ], ignore_index=True)

        del reranker
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if not df_rerank.empty:
    display(df_rerank[df_rerank["layer"].isin({"B_docs", "C_code"})]
            .groupby(["approach", "layer", "language"])["hit@k"].mean().unstack("language").round(3))

[reranker] reranking top-20 on top of hybrid_k60


language                     en     fr
approach          layer               
hybrid_k60+rerank B_docs  0.612  0.575
                  C_code  0.778  0.533

## 7. Per-layer leaderboard

All approaches side-by-side, one table per layer. The negative layer
scores rejection success at increasing K — if it drops as K grows, the
retriever is pulling plausible-but-wrong Kalisio sources for out-of-scope
queries.

In [9]:
from nb05_helpers import leaderboard

df_all = pd.concat([df_dense, df_bm25, df_hybrid, df_rerank], ignore_index=True)

_layers = ("A_symbol", "B_docs", "C_code", "negative")
for layer in _layers:
    print(f"\n=== {layer} ===")
    display(leaderboard(df_all, layer=layer))

_neg_cols = [c for c in ("hit@1", "hit@5", "hit@10") if c in df_all.columns]
df_negative_stress = (
    df_all[df_all["is_negative"]]
    .groupby(["approach", "language"])[_neg_cols]
    .mean()
    .rename(columns={"hit@1": "reject@1", "hit@5": "reject@5", "hit@10": "reject@10"})
    .round(3)
    .sort_values(["language", "reject@5", "reject@1"], ascending=[True, False, False])
)
print("\n=== Negative stress: rejection success by K ===")
display(df_negative_stress)


=== A_symbol ===


language,en,fr,mean
approach,,,
qwen3-0.6b,0.883,0.883,0.883
hybrid_k60,0.883,0.867,0.875
hybrid_k60+rerank,0.883,0.867,0.875
bm25,0.850,0.833,0.841
arctic-l-v2,0.850,0.817,0.833
bge-m3,0.800,0.817,0.808



=== B_docs ===


language,en,fr,mean
approach,,,
qwen3-0.6b,0.725,0.688,0.706
arctic-l-v2,0.638,0.612,0.625
hybrid_k60+rerank,0.612,0.575,0.593
hybrid_k60,0.650,0.500,0.575
bge-m3,0.588,0.525,0.556
bm25,0.400,0.100,0.250



=== C_code ===


language,en,fr,mean
approach,,,
hybrid_k60+rerank,0.778,0.533,0.656
qwen3-0.6b,0.756,0.511,0.634
arctic-l-v2,0.667,0.556,0.612
hybrid_k60,0.667,0.511,0.589
bge-m3,0.600,0.511,0.556
bm25,0.311,0.089,0.200



=== negative ===


language,en,fr,mean
approach,,,
bm25,0.333,0.000,0.166
arctic-l-v2,0.067,0.200,0.134
bge-m3,0.133,0.133,0.133
hybrid_k60,0.067,0.000,0.034
hybrid_k60+rerank,0.067,0.000,0.034
qwen3-0.6b,0.000,0.000,0.000



=== Negative stress: rejection success by K ===


,,reject@1,reject@5,reject@10
approach,language,,,
bm25,en,0.600,0.333,0.067
bge-m3,en,0.400,0.133,0.000
arctic-l-v2,en,0.600,0.067,0.000
hybrid_k60,en,0.533,0.067,0.000
hybrid_k60+rerank,en,0.533,0.067,0.000
qwen3-0.6b,en,0.467,0.000,0.000
arctic-l-v2,fr,0.400,0.200,0.000
bge-m3,fr,0.467,0.133,0.133
qwen3-0.6b,fr,0.600,0.000,0.000


## 8. Cost profile

Throughput, query latency, and index footprint for the surviving dense
models — the budget side of the picture.

In [10]:
from nb05_helpers import measure_throughput, index_size_bytes, query_latency_ms

cost_rows = []
sample_q = query_en[: min(40, len(query_en))]
sample_chunks = chunk_texts[: min(512, len(chunk_texts))]

for key in available_models:
    recipe = ACTIVE_RECIPES[key]
    try:
        model = load_recipe_model(recipe)
    except Exception:
        continue

    thr = measure_throughput(model, recipe, sample_chunks)
    full_cv = encode_corpus(model, recipe, chunk_texts)
    encode_q = lambda q, m=model, r=recipe: encode_queries(m, r, [q])
    lat = query_latency_ms(encode_q, full_cv, sample_q)

    cost_rows.append({
        "model": key,
        "dim": int(full_cv.shape[1]),
        "chunks_per_sec": round(thr["chunks_per_sec"], 1),
        "query_mean_ms": round(lat["mean_ms"], 1),
        "query_p95_ms":  round(lat["p95_ms"], 1),
        "index_mb": round(index_size_bytes(len(chunks), full_cv.shape[1]) / (1024 ** 2), 1),
    })
    del model, full_cv
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

df_cost = pd.DataFrame(cost_rows).set_index("model")
df_cost

,dim,chunks_per_sec,query_mean_ms,query_p95_ms,index_mb
model,,,,,
qwen3-0.6b,1024,91.2,22.9,24.0,38.6
arctic-l-v2,1024,52.7,11.1,11.7,38.6
bge-m3,1024,53.0,11.2,12.0,38.6


## 9. Per-layer winners

Per-layer hit@5 with the best approach highlighted at the top.

In [11]:
from nb05_helpers import per_layer_summary

per_layer = per_layer_summary(df_all)
_winner_layers = [c for c in ("A_symbol", "B_docs", "C_code") if c in per_layer.columns]
winners = {layer: per_layer[layer].idxmax() for layer in _winner_layers}
print("Winners:", winners)
display(per_layer)

Winners: {'A_symbol': 'qwen3-0.6b', 'B_docs': 'qwen3-0.6b', 'C_code': 'hybrid_k60+rerank'}


layer,A_symbol,B_docs,C_code,negative
approach,,,,
arctic-l-v2,0.833,0.625,0.611,0.133
bge-m3,0.808,0.556,0.556,0.133
bm25,0.842,0.250,0.200,0.167
hybrid_k60,0.875,0.575,0.589,0.033
hybrid_k60+rerank,0.875,0.594,0.656,0.033
qwen3-0.6b,0.883,0.706,0.633,0.000


## 10. Persist results

In [12]:
df_all.to_json(OUTPUTS / "nb05_results.json", orient="records", indent=2)
df_cost.to_json(OUTPUTS / "nb05_cost.json", orient="index", indent=2)
if "df_negative_stress" in globals():
    df_negative_stress.to_json(OUTPUTS / "nb05_negative_stress.json", orient="index", indent=2)
print("[save] outputs:")
for p in ("nb05_results.json", "nb05_cost.json", "nb05_negative_stress.json"):
    print(" -", OUTPUTS / p)

[save] outputs:
 - /home/felix/kalisio/knowledge/outputs/nb05_results.json
 - /home/felix/kalisio/knowledge/outputs/nb05_cost.json
 - /home/felix/kalisio/knowledge/outputs/nb05_negative_stress.json


## 11. FR-first decision view

Production traffic is French-leaning, so the headline metric is FR `hit@5`
on the natural-language layers (`B_docs` + `C_code`). The cost-accuracy
frontier and the FR-weighted score below pick the production retriever.

In [13]:
# FR-only per-layer leaderboard
fr = df_all[df_all["language"] == "fr"]
fr_per_layer = (
    fr.groupby(["approach", "layer"])["hit@k"].mean()
    .unstack("layer").round(3)
)
_ordered_cols = [c for c in ("A_symbol", "B_docs", "C_code", "negative") if c in fr_per_layer.columns]
fr_per_layer = fr_per_layer[_ordered_cols]
_bc_cols = [c for c in ("B_docs", "C_code") if c in fr_per_layer.columns]
fr_per_layer["B+C mean"] = fr_per_layer[_bc_cols].mean(axis=1).round(3)
fr_per_layer = fr_per_layer.sort_values("B+C mean", ascending=False)
print("=== FR-only hit@5 by approach × layer ===\n")
display(fr_per_layer)

=== FR-only hit@5 by approach × layer ===



layer,A_symbol,B_docs,C_code,negative,B+C mean
approach,,,,,
qwen3-0.6b,0.883,0.688,0.511,0.000,0.599
arctic-l-v2,0.817,0.612,0.556,0.200,0.584
hybrid_k60+rerank,0.867,0.575,0.533,0.000,0.554
bge-m3,0.817,0.525,0.511,0.133,0.518
hybrid_k60,0.867,0.500,0.511,0.000,0.506
bm25,0.833,0.100,0.089,0.000,0.094


In [14]:
# Cost vs accuracy on FR (B+C) — the dense models only
if not df_cost.empty:
    fr_score = (
        df_all[(df_all["language"] == "fr") & (df_all["layer"].isin(["B_docs", "C_code"]))]
        .groupby("approach")["hit@k"].mean()
    )
    frontier = (
        df_cost.assign(fr_BC_hit5=df_cost.index.map(fr_score).round(3))
        .dropna(subset=["fr_BC_hit5"])
        .sort_values("fr_BC_hit5", ascending=False)
        [["dim", "chunks_per_sec", "query_mean_ms", "index_mb", "fr_BC_hit5"]]
    )
    print("=== Dense models: FR B+C hit@5 vs cost ===\n")
    display(frontier)
else:
    frontier = pd.DataFrame()
    print("(no cost data — dense models did not load)")

=== Dense models: FR B+C hit@5 vs cost ===



,dim,chunks_per_sec,query_mean_ms,index_mb,fr_BC_hit5
model,,,,,
qwen3-0.6b,1024,91.2,22.9,38.6,0.624
arctic-l-v2,1024,52.7,11.1,38.6,0.592
bge-m3,1024,53.0,11.2,38.6,0.520


In [15]:
# FR-weighted score: 60% FR(B+C) + 20% FR(A) + 20% EN(B+C)
def _layer_mean(df, lang, layers):
    sub = df[(df["language"] == lang) & (df["layer"].isin(layers))]
    return sub.groupby("approach")["hit@k"].mean()

fr_bc  = _layer_mean(df_all, "fr", ["B_docs", "C_code"])
fr_a   = _layer_mean(df_all, "fr", ["A_symbol"])
en_bc  = _layer_mean(df_all, "en", ["B_docs", "C_code"])
neg_fr = _layer_mean(df_all, "fr", ["negative"])

scored = (0.6 * fr_bc + 0.2 * fr_a + 0.2 * en_bc).sort_values(ascending=False).round(3)
top_table = pd.DataFrame({
    "weighted_score": scored,
    "FR_B+C": fr_bc.round(3),
    "FR_A":   fr_a.round(3),
    "EN_B+C": en_bc.round(3),
    "FR_neg": neg_fr.round(3),
}).loc[scored.index]
display(top_table.head(10))
print(f"Top-3: {scored.head(3).index.tolist()}")

,weighted_score,FR_B+C,FR_A,EN_B+C,FR_neg
approach,,,,,
qwen3-0.6b,0.698,0.624,0.883,0.736,0.000
arctic-l-v2,0.648,0.592,0.817,0.648,0.200
hybrid_k60+rerank,0.644,0.560,0.867,0.672,0.000
hybrid_k60,0.607,0.504,0.867,0.656,0.000
bge-m3,0.594,0.520,0.817,0.592,0.133
bm25,0.298,0.096,0.833,0.368,0.000


Top-3: ['qwen3-0.6b', 'arctic-l-v2', 'hybrid_k60+rerank']


<!-- nb05.xlingual.v1 -->

## 12. Cross-lingual experiment — would French docs close the FR→EN gap?

**Motivation.** Section 11 makes it clear that French queries trail English by
several points on the natural-language layers (`B_docs`: FR 0.688 vs EN 0.725
on the leader; `C_code`: FR 0.511 vs EN 0.756). The Kalisio corpus is currently
English-only, so every French query has to bridge the FR↔EN embedding gap
before matching a passage. The natural follow-up question:

> If a future Kalisio project shipped French documentation, would scoring it
> monolingually (FR query → FR docs) recover that lost ground?

**Why this matters in production.** A French-first deployment, or even a mixed
EN+FR knowledge base, changes the retrieval calculus. Three numbers we want:

1. The cross-lingual penalty as it stands today (FR → EN corpus).
2. What FR → FR retrieval would look like on a comparable corpus.
3. Whether a mixed bilingual index gains or loses anything over either
   monolingual setup.

**Test corpus.** We use [`kalisio/dok`](https://github.com/kalisio/dok) — the
user-facing documentation for Kalisio Crisis. It ships a parallel `docs/`
(English) and `docs/fr/` (French) tree where every English page has a
hand-written French counterpart of comparable length and content. **24 EN
files mirror 24 FR files** (≈264 KB of text), giving three corpus variants
from real human-translated material rather than machine-translated synthetic
data:

- `EN-only`: 24 English docs (~108 chunks).
- `FR-only`: 24 French docs (~108 chunks).
- `bilingual`: both, in the same index (~216 chunks).

**Gold set.** 45 natural-language questions written from the actual content of
the docs (see `experiments/nb05_embedding_eval/nb05_xlingual_gold.json`). Each query has
an EN/FR pair. The mix is deliberate:

- **~11 lexical anchors** — identical proper nouns / numbers / URLs across
  languages (`Akt'n'Map`, `status.kalisio.com`, iOS `16.4`, `ARPEGE/AROME/GFS`,
  `WMS/WFS/TMS/WMTS`, `GoogleMaps/Waze`, `20cm/pixel`). Small expected
  cross-lingual gap — these mostly test embedding noise floor.
- **~8 translation-required terms** — EN/FR diverge sharply (`logbook` ↔
  `main courante`, `tag` ↔ `étiquette`, `workflow` ↔ `processus`, Kanban
  column names `To do/Doing/Done` ↔ `À faire/En cours/Clôturés`). Large
  expected cross-lingual gap — these stress the embedding's cross-lingual
  alignment.
- **~25 general concept questions** — paraphrased questions about features
  described in both languages.
- **1 outlier** flagged `richer_in_fr` (`Q-038` Hub'Eau): the FR doc has a
  paragraph where EN has one sentence. Isolated for separate analysis so it
  doesn't conflate "FR→FR wins because monolingual" with "FR→FR wins because
  FR doc is richer."

**Experiment matrix.** Each of the three production-candidate dense models
(`qwen3-0.6b`, `arctic-l-v2`, `bge-m3`) is evaluated on four settings:

| Setting       | Corpus     | Query | What it measures                                |
|---------------|------------|-------|--------------------------------------------------|
| `EN→EN`       | 24 EN docs | EN    | Monolingual EN ceiling                           |
| `FR→EN`       | 24 EN docs | FR    | Cross-lingual penalty (today's nb05 setup)       |
| `FR→FR`       | 24 FR docs | FR    | Monolingual FR — the hypothesis under test       |
| `FR→bi`       | 48 mixed   | FR    | Realistic production: both langs in same index   |

Headline metric is `hit@5` on natural-language queries (this corpus has no
`A_symbol`/`C_code` content — it's pure user docs, equivalent to nb05's
`B_docs` layer). Expected pattern if the hypothesis holds:

- `FR→FR` ≈ `EN→EN` (monolingual symmetric).
- `FR→EN` < `FR→FR` by 5–15 points — the cross-lingual gap.
- `FR→bi` ≥ `FR→FR` (FR doc still in the index, retriever picks it).

If `FR→FR` doesn't catch up to `EN→EN`, something other than the
cross-lingual gap is in play — corpus differences, FR doc quality, or model
imbalance.


### 12.1 Load the bilingual `dok` corpus and the cross-lingual gold set

The dok docs live under `experiments/nb05_embedding_eval/nb05_xlingual_data/docs/`
(landed there alongside the rest of the nb05 fixtures so the main `data/`
corpus stays pristine — adding 48 unrelated user docs would shift nb05's
earlier numbers in §3). The loader reuses the JS/Vue RAG profile but reinstates
`tutorials/` (a real content directory in dok, only excluded by default for the
JS/Vue profile).

In [16]:
from nb05_xlingual_helpers import load_dok_corpus, load_xlingual_gold

DOK_DATA_ROOT = ROOT / "experiments" / "nb05_embedding_eval" / "nb05_xlingual_data"
DOK_GOLD_PATH = ROOT / "experiments" / "nb05_embedding_eval" / "nb05_xlingual_gold.json"

dok = load_dok_corpus(DOK_DATA_ROOT)
print(f"[dok corpus] EN chunks={len(dok['en_chunks'])}  FR chunks={len(dok['fr_chunks'])}  total={len(dok['bi_sources'])}")
print(f"[dok corpus] EN files={len(set(dok['en_sources']))}  FR files={len(set(dok['fr_sources']))}")

dok_gold = load_xlingual_gold(DOK_GOLD_PATH, dok["en_sources"], dok["fr_sources"])
xl_queries = dok_gold["queries"]
en_gold, fr_gold = dok_gold["en_gold"], dok_gold["fr_gold"]
print(f"[gold]      {len(xl_queries)} queries  (schema={dok_gold['version']})")
print(f"[gold]      all gold paths verified in corpus")


[dok corpus] EN chunks=108  FR chunks=108  total=216
[dok corpus] EN files=24  FR files=24
[gold]      45 queries  (schema=nb05.xlingual.v1)
[gold]      all gold paths verified in corpus


### 12.2 Dense bake-off across the 4×3 matrix

For each model we encode the EN corpus once, the FR corpus once, and reuse
the union for the bilingual setting (`bi_vecs = concat(en_vecs, fr_vecs)`).
Each query is encoded twice (EN + FR text). `hit@5` is computed against the
appropriate gold path set for each setting. The cell mirrors the management
discipline from §3 — load model → encode → score → free GPU.


In [17]:
from nb05_xlingual_helpers import run_xlingual_bakeoff

XL_K = 5
df_xl = run_xlingual_bakeoff(
    queries=xl_queries,
    corpus=dok,
    en_gold=en_gold,
    fr_gold=fr_gold,
    available_models=available_models,  # the 3 models that loaded successfully in §3
    active_recipes=ACTIVE_RECIPES,
    k=XL_K,
)
print(f"\n[xl] {len(df_xl)} rows  ({df_xl['model'].nunique()} models × {df_xl['query_id'].nunique()} queries)")
df_xl.head()


[xl] qwen3-0.6b -> loading
[xl] qwen3-0.6b -> encoding EN (108) and FR (108) chunks
[xl] arctic-l-v2 -> loading
[xl] arctic-l-v2 -> encoding EN (108) and FR (108) chunks
[xl] bge-m3 -> loading
[xl] bge-m3 -> encoding EN (108) and FR (108) chunks

[xl] 135 rows  (3 models × 45 queries)


,model,query_id,section,richer_in_fr,EN→EN,FR→EN,FR→FR,FR→bi
0,qwen3-0.6b,Q-001,about,False,1,1,1,1
1,qwen3-0.6b,Q-002,about,False,1,1,1,1
2,qwen3-0.6b,Q-003,about,False,1,1,1,1
3,qwen3-0.6b,Q-004,about,False,1,1,1,1
4,qwen3-0.6b,Q-005,quickstart,False,1,1,1,1


### 12.3 Cross-lingual leaderboard

`hit@5` averaged across the full 45-query gold set, per model × setting.
The interesting columns are the deltas on the right:

- `gap (FR→EN − FR→FR)` is the cross-lingual penalty: positive means
  FR→FR beats FR→EN, which is the hypothesis under test. Negative would
  mean FR docs hurt — investigate corpus quality if so.
- `FR→FR vs EN→EN` shows whether FR monolingual catches up to the EN
  monolingual ceiling. ~0 means yes; large negative means FR doc / FR
  embedding alignment is the bottleneck, not the cross-lingual gap.
- `FR→bi vs FR→FR` checks whether bilingual indexing is free, costly, or
  a small win.


In [18]:
from nb05_xlingual_helpers import xlingual_leaderboard

xl_per_model = xlingual_leaderboard(df_xl, available_models)
print(f"=== hit@{XL_K} on dok ({len(xl_queries)} queries) ===\n")
display(xl_per_model)

mean_gap = xl_per_model["gap (FR→FR − FR→EN)"].mean().round(3)
mean_ceiling_loss = xl_per_model["FR→FR − EN→EN"].mean().round(3)
print(f"\nMean cross-lingual gap recovered by FR→FR: {mean_gap:+.3f} hit@{XL_K}")
print(f"Mean residual gap to EN→EN ceiling:        {mean_ceiling_loss:+.3f} hit@{XL_K}")


=== hit@5 on dok (45 queries) ===



,EN→EN,FR→EN,FR→FR,FR→bi,gap (FR→FR − FR→EN),FR→FR − EN→EN,FR→bi − FR→FR
model,,,,,,,
qwen3-0.6b,0.911,0.867,0.889,0.889,0.022,-0.022,0.000
arctic-l-v2,0.889,0.867,0.889,0.867,0.022,0.000,-0.022
bge-m3,0.867,0.844,0.911,0.911,0.067,0.044,0.000



Mean cross-lingual gap recovered by FR→FR: +0.037 hit@5
Mean residual gap to EN→EN ceiling:        +0.007 hit@5


### 12.4 Where does the gap actually live?

The 45 questions weren't designed equally — some have language-agnostic
anchors (`Akt'n'Map`, `WMS`, `iOS 16.4`) where cross-lingual retrieval
should already work, and some have term-mismatch (`logbook` ↔ `main
courante`, `tag` ↔ `étiquette`) where the cross-lingual gap should be
largest. We can label each query and re-aggregate.

The `Q-038` outlier (`richer_in_fr=True`, Hub'Eau) is excluded from the
"clean" view — its FR→FR advantage conflates cross-lingual gap with
content-richness gap.


In [24]:
from nb05_xlingual_helpers import xlingual_category_view

cat_view, cat_gap = xlingual_category_view(df_xl)
print("=== hit@5 per query category × model (richer_in_fr excluded) ===\n")
display(cat_view)



=== hit@5 per query category × model (richer_in_fr excluded) ===



EN→EN  FR→EN  FR→FR  FR→bi
category    model                                  
anchor      arctic-l-v2  1.000  1.000  1.000  1.000
            bge-m3       1.000  1.000  1.000  1.000
            qwen3-0.6b   1.000  1.000  1.000  1.000
general     arctic-l-v2  0.840  0.800  0.840  0.800
            bge-m3       0.840  0.800  0.880  0.880
            qwen3-0.6b   0.880  0.840  0.840  0.840
translation arctic-l-v2  0.875  0.875  0.875  0.875
            bge-m3       0.750  0.750  0.875  0.875
            qwen3-0.6b   0.875  0.750  0.875  0.875

<!-- nb05.augment.v1 -->

## 13. Closing the FR gap — glossary vs LLM query translation

The per-query error analysis on §3's results split FR failures cleanly into two
populations:

| Does the FR query carry an EN identifier anchor? | n   | FR-lose rate | net Δ |
|---|---|---|---|
| Yes (`composant KChart`, `activité MapActivity`, …) | 114 | **10.5%** | +6  |
| No  (concept paraphrased, no project-specific term) | 71  | **32.4%** | +25 |

The shortfall is concentrated in code targets (`.vue` 36%, `.js` 27%) and the
`C_code` layer (50% of `.vue`-targeted C_code queries lose), where chunks are
dominated by English identifiers like `mailer`, `EventLog`, `MapActivity`,
`KImportLayer`. A French query asking *"où est implémenté le service d'envoi
d'emails ?"* has no path to `mailer.service.js` because the identifier
`mailer` exists only in English. Multilingual embeddings translate
`envoi d'emails` ≈ `email sending` — but not ≈ `mailer`, which is project
naming, not standard English.

The §12 bilingual experiment showed that when the corpus has parallel
EN/FR content with shared anchors (proper nouns, URLs, brand names),
the cross-lingual gap is ≈0. So the §11 FR shortfall is **not an embedding
problem** — it's a **terminology coverage problem**. Two ways to bridge it:

1. **Glossary augmentation** — deterministic mapping of FR
   concept-phrases to EN project terms (`envoi d'emails → mailer`). Append
   the EN terms to the query so the embedding gets both signals.
2. **LLM query translation** — pass the FR query through Claude Haiku,
   embed the EN translation. Generic, no maintenance, but ignorant of
   project-specific naming.

This section runs four query variants on the same corpus, encoded by the
three dense models from §3:

| Variant         | What's embedded                                       |
|-----------------|--------------------------------------------------------|
| `FR_orig`       | Original French query (current §11 baseline)           |
| `FR_glossary`   | Original FR + EN terms from glossary, in parens        |
| `FR_LLM`        | Local Ollama (qwen2.5:7b) English translation (replaces the FR)   |
| `FR_combined`   | Original FR + glossary terms + LLM translation         |

Headline question: does either path recover the +31-net-delta FR shortfall
from §11, and which fails on which kind of query?

> **Note on running this section.** The glossary path is deterministic and
> runs out-of-the-box. The LLM path requires `ANTHROPIC_API_KEY` to be set
> in the environment; if missing, those cells degrade gracefully and skip.
> Translation results are cached on disk
> (`experiments/nb05_embedding_eval/nb05_xlingual_translation_cache.json`) so repeat
> runs cost nothing.


### 13.1 Build the project glossary

The glossary is a hand-curated list of FR phrase patterns → EN project terms.
Each entry was selected from the §3 FR-lose set (the 35 queries where FR
underperformed EN on at least one model). Patterns are regex, applied
case-insensitively. When a pattern matches, its EN expansion is appended in
parens — **not substituted** — so the original FR semantics are preserved
and added to, not replaced.

Curation principle: keep entries concept→identifier (`envoi d'emails` →
`mailer`), skip generic word-level translations (`carte` → `map`) that
multilingual embeddings already handle. ~25 entries was enough to touch
most failure cases; the marginal entry past that adds noise risk without
much coverage.


In [20]:
from nb05_glossary import PROJECT_GLOSSARY
from nb05_helpers import augment_fr_query_with_project_glossary as augment_fr


# Coverage check: how many of the 35 FR-loses queries does the glossary touch?
df_dense_fr = df_dense[df_dense["language"] == "fr"]
fr_loses_ids = set()
for qid in {q.id for q in queries if q.layer != "negative"}:
    en_hits = df_dense[(df_dense["query_id"] == qid) & (df_dense["language"] == "en")]["hit@k"].sum()
    fr_hits = df_dense_fr[df_dense_fr["query_id"] == qid]["hit@k"].sum()
    if en_hits > fr_hits:
        fr_loses_ids.add(qid)

print(f"glossary entries: {len(PROJECT_GLOSSARY)}")
print(f"FR-loses queries (delta>0 across the 3 dense models): {len(fr_loses_ids)}")

touched_loses = []
touched_other = []
for q in queries:
    if q.layer == "negative":
        continue
    aug_query, augs = augment_fr(q.fr)
    if augs:
        if q.id in fr_loses_ids:
            touched_loses.append((q.id, q.layer, q.fr[:60], augs))
        else:
            touched_other.append(q.id)

print(f"queries touched by glossary: {len(touched_loses) + len(touched_other)}")
print(f"  of which FR-loses: {len(touched_loses)} / {len(fr_loses_ids)}")
print(f"  of which other:    {len(touched_other)}")

print("\nFR-loses queries touched by glossary:")
for qid, layer, fr_short, augs in touched_loses[:25]:
    print(f"  [{qid}] {layer:<8} {fr_short!r:<62} -> {augs}")


glossary entries: 39
FR-loses queries (delta>0 across the 3 dense models): 35
queries touched by glossary: 37
  of which FR-loses: 25 / 35
  of which other:    12

FR-loses queries touched by glossary:
  [B-013] B_docs   "Comment internationaliser l'interface ?"                      -> ['i18n internationalize locale']
  [B-017] B_docs   "Comment intercepter une requête avant qu'elle n'arrive en ba" -> ['hook query intercept']
  [B-025] B_docs   "Comment échantillonner les valeurs d'une couche météo ?"      -> ['weather probe layer']
  [B-039] B_docs   'Les routes de l’application sont déclarées où dans les proje' -> ['router routes']
  [B-044] B_docs   'Je veux comprendre les formulaires générés depuis les schéma' -> ['KForm schema form']
  [B-060] B_docs   'Les visites guidées dans les applis Kalisio, elles sont décl' -> ['tour KTour']
  [B-061] B_docs   'Dans Crisis, comment s’articulent les événements, les plans,' -> ['EventsActivity PlansActivity OrganisationMenu PlanMenu']
  [B-06

### 13.2 Generate the four FR query variants

Two transformations to apply across all 200 gold queries (negatives included
for symmetry, even though hit@k on negatives is always 0):

- **Glossary**: deterministic, runs immediately.
- **LLM**: one Ollama call per query against `qwen2.5:7b` on the local
  network. 


In [21]:
from nb05_augment_helpers import LLMTranslator

translator = LLMTranslator(
    cache_path=ROOT / "experiments" / "nb05_embedding_eval" / "nb05_xlingual_translation_cache.json",
)

# Build the four variants
fr_orig: list[str]      = [q.fr for q in queries]
fr_glossary: list[str]  = [augment_fr(q.fr)[0] for q in queries]
fr_llm: list[str | None] = translator.translate_all(fr_orig)
fr_combined: list[str]  = [
    f"{augment_fr(o)[0]} {l}" if l else augment_fr(o)[0]
    for o, l in zip(fr_orig, fr_llm)
]

n_llm_ok = sum(1 for t in fr_llm if t is not None)
print(f"\n[variants] orig={len(fr_orig)} glossary={len(fr_glossary)} llm={n_llm_ok}/{len(fr_llm)} combined={len(fr_combined)}")

print("\nsample variants:")
for qi in [11, 12, 16, 31]:  # B-013, B-014, B-017, B-032 from earlier analysis
    if qi >= len(queries): continue
    q = queries[qi]
    print(f"\n  [{q.id}] layer={q.layer}")
    print(f"    FR_orig:     {q.fr}")
    aug_q, augs = augment_fr(q.fr)
    print(f"    FR_glossary: {aug_q if augs else '(no glossary match)'}")
    print(f"    FR_LLM:      {fr_llm[qi] if fr_llm[qi] else '(skipped)'}")


[llm] 20 translations done
[llm] 40 translations done
[llm] 60 translations done
[llm] 80 translations done
[llm] 100 translations done
[llm] 120 translations done
[llm] 140 translations done
[llm] 160 translations done
[llm] 180 translations done
[llm] 200 translations done
[llm] 200 new translations  |  0 cache hits  |  0 errors  |  model=qwen2.5:7b

[variants] orig=200 glossary=200 llm=200/200 combined=200

sample variants:

  [A-012] layer=A_symbol
    FR_orig:     composable useCollection
    FR_glossary: (no glossary match)
    FR_LLM:      composable useCollection

  [A-013] layer=A_symbol
    FR_orig:     composable useCatalog
    FR_glossary: (no glossary match)
    FR_LLM:      composable useCatalog

  [A-017] layer=A_symbol
    FR_orig:     service authorisations
    FR_glossary: (no glossary match)
    FR_LLM:      authorisation service

  [A-032] layer=A_symbol
    FR_orig:     fonction utils.features
    FR_glossary: (no glossary match)
    FR_LLM:      function utils.fea

### 13.3 Re-encode and score the variants

Each model encodes the corpus once, then encodes the four query variants
and scores each against the original gold. The corpus encoding is the
expensive part (~30s per model on this machine); query encoding for
all four variants is fast.

The LLM and combined variants gracefully fall back to baseline FR for
queries where translation was skipped — so the metric is comparable
across variants even when the API was unavailable for some entries.


In [22]:
VARIANTS = ("FR_orig", "FR_glossary", "FR_LLM", "FR_combined")
variant_texts = {
    "FR_orig":     fr_orig,
    "FR_glossary": fr_glossary,
    "FR_LLM":      [t if t else o for t, o in zip(fr_llm, fr_orig)],         # fallback to FR
    "FR_combined": fr_combined,
}
# Track whether LLM translation was available (for honest reporting)
llm_available_mask = np.array([t is not None for t in fr_llm])

aug_dfs: list[pd.DataFrame] = []

for key in available_models:
    recipe = ACTIVE_RECIPES[key]
    print(f"\n[aug] {key}: load + encode corpus")
    model = load_recipe_model(recipe)
    cv = encode_corpus(model, recipe, chunk_texts)

    for variant in VARIANTS:
        qv = encode_queries(model, recipe, variant_texts[variant])
        ranks = dense_rank(qv, cv)
        df = evaluate_ranks(ranks, queries, chunk_sources, language="fr", approach=f"{key}::{variant}")
        df["model"] = key
        df["variant"] = variant
        aug_dfs.append(df)
        print(f"  {variant:<13} hit@5={df['hit@5'].mean():.3f}")

    del model, cv
    gc.collect()
    if torch.cuda.is_available():
        try: torch.cuda.empty_cache()
        except Exception: pass

df_aug = pd.concat(aug_dfs, ignore_index=True)
print(f"\n[aug] {len(df_aug)} rows  ({df_aug['model'].nunique()} models × {len(VARIANTS)} variants × {df_aug['query_id'].nunique()} queries)")



[aug] qwen3-0.6b: load + encode corpus
  FR_orig       hit@5=0.655
  FR_glossary   hit@5=0.750
  FR_LLM        hit@5=0.665
  FR_combined   hit@5=0.725

[aug] arctic-l-v2: load + encode corpus
  FR_orig       hit@5=0.630
  FR_glossary   hit@5=0.680
  FR_LLM        hit@5=0.630
  FR_combined   hit@5=0.680

[aug] bge-m3: load + encode corpus
  FR_orig       hit@5=0.580
  FR_glossary   hit@5=0.650
  FR_LLM        hit@5=0.625
  FR_combined   hit@5=0.655

[aug] 2400 rows  (3 models × 4 variants × 200 queries)


### 13.4 Headline — does augmentation close the FR gap?

Per-model `hit@5` averaged across the positive (non-negative) gold subset.
Compare against §3's EN baseline (the ceiling we'd love FR to reach).


In [23]:
en_baseline = (
    df_dense[(df_dense["language"] == "en")]
    .query("not is_negative")
    .groupby("approach")["hit@5"].mean().round(3)
    .reindex(available_models)
)

headline = (
    df_aug.query("not is_negative")
    .groupby(["model", "variant"])["hit@5"].mean()
    .unstack("variant").round(3)
    .reindex(available_models)
    [list(VARIANTS)]
)
headline.insert(0, "EN_orig", en_baseline)
headline["Δ glossary"] = (headline["FR_glossary"] - headline["FR_orig"]).round(3)
headline["Δ LLM"]      = (headline["FR_LLM"]      - headline["FR_orig"]).round(3)
headline["Δ combined"] = (headline["FR_combined"] - headline["FR_orig"]).round(3)
headline["EN−combined"] = (headline["EN_orig"]    - headline["FR_combined"]).round(3)

print("=== Per-model hit@5 across query variants (positive queries only) ===\n")
display(headline)

# Per-layer breakdown — the §11 shortfall was concentrated in C_code
print("\n=== hit@5 by layer × variant (averaged over the 3 dense models) ===\n")
layer_view = (
    df_aug.query("not is_negative")
    .groupby(["layer", "variant"])["hit@5"].mean()
    .unstack("variant").round(3)
    [list(VARIANTS)]
    .reindex(["A_symbol", "B_docs", "C_code"])
)
en_layer = (
    df_dense.query("language=='en' and not is_negative")
    .groupby(["layer", "approach"])["hit@5"].mean()
    .groupby("layer").mean().round(3)
    .reindex(["A_symbol", "B_docs", "C_code"])
)
layer_view.insert(0, "EN_orig", en_layer)
layer_view["Δ combined"] = (layer_view["FR_combined"] - layer_view["FR_orig"]).round(3)
layer_view["EN−combined"] = (layer_view["EN_orig"] - layer_view["FR_combined"]).round(3)
display(layer_view)

# LLM coverage caveat
n_llm_skipped = (~llm_available_mask).sum()
if n_llm_skipped:
    print(f"\n[caveat] FR_LLM (and FR_combined) used FR fallback for {n_llm_skipped} queries where Claude was unavailable.")


=== Per-model hit@5 across query variants (positive queries only) ===



variant,EN_orig,FR_orig,FR_glossary,FR_LLM,FR_combined,Δ glossary,Δ LLM,Δ combined,EN−combined
model,,,,,,,,,
qwen3-0.6b,0.784,0.708,0.811,0.719,0.784,0.103,0.011,0.076,0.000
arctic-l-v2,0.714,0.665,0.719,0.670,0.730,0.054,0.005,0.065,-0.016
bge-m3,0.659,0.616,0.692,0.665,0.703,0.076,0.049,0.087,-0.044



=== hit@5 by layer × variant (averaged over the 3 dense models) ===



variant,EN_orig,FR_orig,FR_glossary,FR_LLM,FR_combined,Δ combined,EN−combined
layer,,,,,,,
A_symbol,0.844,0.839,0.839,0.839,0.872,0.033,-0.028
B_docs,0.650,0.608,0.675,0.617,0.667,0.059,-0.017
C_code,0.674,0.526,0.726,0.600,0.689,0.163,-0.015
